# BookWise AI — Phase 1 Model Training

This notebook trains a personalized book recommendation model using the Kaggle Book Recommendation Dataset.

Final output files will be saved into `../models/` and used later by the FastAPI backend.

## Step 1: Import libraries

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

## Step 2: Set paths

In [ ]:
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'

BOOKS_FILE = DATA_DIR / 'Books.csv'
RATINGS_FILE = DATA_DIR / 'Ratings.csv'
USERS_FILE = DATA_DIR / 'Users.csv'

## Step 3: Load dataset

In [ ]:
books = pd.read_csv(BOOKS_FILE, low_memory=False)
ratings = pd.read_csv(RATINGS_FILE, low_memory=False)
users = pd.read_csv(USERS_FILE, low_memory=False)

books.shape, ratings.shape, users.shape

## Step 4: Clean books data

In [ ]:
books = books[[
    'ISBN', 'Book-Title', 'Book-Author',
    'Year-Of-Publication', 'Publisher', 'Image-URL-M'
]].copy()

books.rename(columns={
    'ISBN': 'isbn',
    'Book-Title': 'title',
    'Book-Author': 'author',
    'Year-Of-Publication': 'year',
    'Publisher': 'publisher',
    'Image-URL-M': 'image_url'
}, inplace=True)

books['title'] = books['title'].astype(str).str.strip()
books['author'] = books['author'].astype(str).str.strip()
books['publisher'] = books['publisher'].astype(str).str.strip()
books['year'] = pd.to_numeric(books['year'], errors='coerce').fillna(0).astype(int)
books['image_url'] = books['image_url'].fillna('')

books.dropna(subset=['isbn', 'title', 'author'], inplace=True)
books.drop_duplicates(subset=['isbn'], inplace=True)
books.drop_duplicates(subset=['title'], keep='first', inplace=True)

books.head()

## Step 5: Clean ratings data

In [ ]:
ratings = ratings[['User-ID', 'ISBN', 'Book-Rating']].copy()
ratings.rename(columns={
    'User-ID': 'user_id',
    'ISBN': 'isbn',
    'Book-Rating': 'rating'
}, inplace=True)

ratings['rating'] = pd.to_numeric(ratings['rating'], errors='coerce')
ratings.dropna(subset=['user_id', 'isbn', 'rating'], inplace=True)
ratings = ratings[(ratings['rating'] >= 0) & (ratings['rating'] <= 10)]

ratings.head()

## Step 6: Build popular books table

In [ ]:
merged = ratings.merge(books, on='isbn')

popular_books = (
    merged.groupby(['title', 'author', 'image_url'], as_index=False)
    .agg(num_ratings=('rating', 'count'), avg_rating=('rating', 'mean'))
)

popular_books = popular_books[popular_books['num_ratings'] >= 50]
popular_books['avg_rating'] = popular_books['avg_rating'].round(2)
popular_books = popular_books.sort_values(
    by=['avg_rating', 'num_ratings'], ascending=False
).head(100)

popular_books.head()

## Step 7: Prepare pivot table for collaborative filtering

In [ ]:
user_rating_counts = merged.groupby('user_id')['rating'].count()
active_users = user_rating_counts[user_rating_counts >= 50].index
filtered = merged[merged['user_id'].isin(active_users)]

book_rating_counts = filtered.groupby('title')['rating'].count()
selected_books = book_rating_counts[book_rating_counts >= 20].index
filtered = filtered[filtered['title'].isin(selected_books)]

top_titles = (
    filtered.groupby('title')['rating']
    .count()
    .sort_values(ascending=False)
    .head(1200)
    .index
)
filtered = filtered[filtered['title'].isin(top_titles)]

pivot_table = filtered.pivot_table(
    index='title',
    columns='user_id',
    values='rating',
    fill_value=0
)

pivot_table.shape

## Step 8: Train cosine similarity model

In [ ]:
similarity_scores = cosine_similarity(pivot_table)
similarity_scores.shape

## Step 9: Create book details table

In [ ]:
book_details = (
    filtered.groupby('title', as_index=False)
    .agg(
        author=('author', 'first'),
        year=('year', 'first'),
        publisher=('publisher', 'first'),
        image_url=('image_url', 'first'),
        num_ratings=('rating', 'count'),
        avg_rating=('rating', 'mean'),
    )
)
book_details['avg_rating'] = book_details['avg_rating'].round(2)
book_details.head()

## Step 10: Test recommendation

In [ ]:
def recommend(book_title, top_n=5):
    if book_title not in pivot_table.index:
        return f'Book not found: {book_title}'

    book_index = np.where(pivot_table.index == book_title)[0][0]
    distances = similarity_scores[book_index]
    similar_items = sorted(
        list(enumerate(distances)),
        key=lambda item: item[1],
        reverse=True
    )[1:top_n+1]

    results = []
    for idx, score in similar_items:
        title = pivot_table.index[idx]
        detail = book_details[book_details['title'] == title].head(1)
        row = detail.iloc[0]
        results.append({
            'title': title,
            'author': row['author'],
            'avg_rating': row['avg_rating'],
            'similarity_score': round(float(score), 3)
        })
    return pd.DataFrame(results)

sample_book = pivot_table.index[0]
sample_book, recommend(sample_book)

## Step 11: Save model artifacts

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)

artifacts = {
    'similarity.pkl': similarity_scores,
    'pivot_table.pkl': pivot_table,
    'books.pkl': books,
    'popular_books.pkl': popular_books,
    'book_details.pkl': book_details,
}

for filename, obj in artifacts.items():
    with open(MODELS_DIR / filename, 'wb') as file:
        pickle.dump(obj, file)

list(MODELS_DIR.glob('*.pkl'))